# P70 — Consistencia en redes de relaciones

## 1. Título y paper

**Paper:** *Consistency in Networks of Relations*  
**Autoría:** Alan K. Mackworth  
**Año y venue:** 1977 · Artificial Intelligence, 8(1), 99–118  
**Nivel:** L3 · **Motor:** `arco_consistencia`  
**Ficha completa:** [`P70_arco_consistencia`](../../papers/foundational/P70_arco_consistencia/README.md)

**Hito:** Convierte la propagación de restricciones en un preproceso con nombre y algoritmo: podar dominios antes de buscar, no mientras se busca.

- [doi:10.1016/0004-3702(77)90007-8](https://doi.org/10.1016/0004-3702%2877%2990007-8)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: El retroceso cronológico repetía una y otra vez el mismo descubrimiento: que cierto valor era incompatible con sus vecinos. La información se hallaba y se tiraba en cada rama.
2. Ejecutar una implementación mínima de la propuesta: Hacer la red consistente de arco antes de asignar nada: eliminar de cada dominio los valores sin compañero legal en algún vecino, y repropagar en cascada. Los algoritmos AC-1, AC-2 y AC-3 formalizan el procedimiento.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- Waltz (1972), filtrado de etiquetas en escenas
- Montanari (1974), redes de restricciones


## 4. Intuición

Si un valor de una variable no tiene ningún compañero legal en su vecina, ese valor no puede estar en ninguna solución. Descubrirlo cuesta una comprobación local, y hacerlo **antes** de buscar ahorra descubrirlo una y otra vez en cada rama del retroceso.


## 5. Concepto mínimo

```text
Arco (x, y) consistente ⟺ todo valor de dom(x) tiene algún compañero legal en dom(y)

AC-3:  cola ← todos los arcos
       mientras la cola no esté vacía:
           (x,y) ← sacar
           si podar dom(x) cambia algo → volver a encolar los arcos (z,x)

Coste O(e·d³). No decide satisfacibilidad: reduce dominios.
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('arco_consistencia', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuántos valores elimina AC-3 antes de asignar nada?
2. ¿Cuántos nodos visita el retroceso con y sin esa poda?
3. ¿Devuelven las dos búsquedas la misma solución?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('arco_consistencia', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('arco_consistencia', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

AC-3 elimina **60 de los 72** valores iniciales sin asignar nada. Después el retroceso resuelve con **7 nodos y 0 retrocesos**, frente a **233 nodos y 226 retrocesos** sin podar. Y la solución es la misma: la consistencia de arco no descarta soluciones, descarta valores que no participan en ninguna.


## 10. Comentario pedagógico

El resultado fuerte está en la última línea de la evidencia: en una red con estructura de **árbol** —como esta cadena— la consistencia de arco deja la búsqueda sin retrocesos. En una red con ciclos ayuda pero no lo garantiza, y ahí es donde vive la investigación posterior en descomposición de restricciones.


## 11. Error o anti-patrón deliberado

Anti-patrón: creer que si AC-3 deja todos los dominios no vacíos, hay solución.


In [ ]:
print('AC-3 elimina valores localmente inconsistentes. Nada mas.')
print('Puede dejar todos los dominios llenos y que el problema no tenga solucion.')
print('Consistencia local no implica consistencia global.')

## 12. Corrección

Lo que sí garantiza y lo que no:


In [ ]:
r = run_paper_lab('arco_consistencia', seed=7)['result']
print('valores podados      :', r['valores_podados'], 'de', r['tamano_del_espacio'] and 72)
print('sin AC-3             :', r['backtracking_sin_ac3']['nodos'], 'nodos ·',
      r['backtracking_sin_ac3']['retrocesos'], 'retrocesos')
print('con AC-3             :', r['backtracking_con_ac3']['nodos'], 'nodos ·',
      r['backtracking_con_ac3']['retrocesos'], 'retrocesos')
print('misma solucion       :', r['misma_solucion'])

## 13. Desafío guiado

Mira los dominios tras AC-3 y explica por qué la primera variable pierde diez de sus doce valores sin que se haya asignado nada.


In [ ]:
r = run_paper_lab('arco_consistencia', seed=3)['result']
show(r)

## 14. Desafío autónomo

Modela el coloreado del mapa de Australia como CSP y aplica AC-3. Comprueba que ahí la poda ayuda menos y explica por qué: la red tiene ciclos y la propiedad del árbol no vale.


## 15. Evidencia de aprendizaje

Guarda la comparación de nodos y retrocesos con y sin poda, y tu enunciado de qué garantiza la consistencia de arco.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P70_arco_consistencia/README.md) · evaluación formal: [`assessments/papers/P70_arco_consistencia.md`](../../assessments/papers/P70_arco_consistencia.md)


## 16. Cierre

Las restricciones ya se propagan. Falta la pieza que permite que dos sistemas distintos hablen del mismo mundo: un acuerdo explícito sobre qué significa cada término.


## 17. Conexión con el siguiente hito

- P65

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
